In [1]:
import pandas as pd
import ast
import plotly.express as px
import plotly.offline as py

# --- 1. CONFIGURAÇÕES E CARREGAMENTO DE DADOS ---
DATA_PATH = "data/macrotopics_pred.csv"
OUTPUT_FILE = "analise_temporal_interativa.html"

# Mapeamento de Cores para manter a consistência visual (opcional)
MACRO_COLORS = {
    "American Politics": "#6AF5FF",
    "Foreign Policy": "#76FF7F",
    "Crime, Security and Justice": "#FF66CC",
    "Environment": "#522222",
    "Health": "#14712A",
    "Entertainment": "#ECF988",
    "Hobbies": "#DA0000",
    "Religion": "#962EAA",
    "Technology": "#0000FF",
    "War": "#5C5C61",
    "Others": "#FF9900"
}

def parse_occurrences(x):
    """Função para converter a string de ocorrências em lista."""
    try:
        return ast.literal_eval(x)
    except:
        return []

def preprocess_data(file_path):
    """Carrega, processa e agrupa os dados."""
    try:
        df_final = pd.read_csv(file_path)
    except FileNotFoundError:
        print(f"Erro: O arquivo de dados '{file_path}' não foi encontrado.")
        return None

    df_final['occ_list'] = df_final['occurrences'].apply(parse_occurrences)
    
    # Explode o dataframe para ter uma linha por ocorrência
    df_exploded = df_final.explode('occ_list', ignore_index=True)
    df_exploded['date'] = df_exploded['occ_list'].apply(
        lambda x: x.get('date') if isinstance(x, dict) and 'date' in x else None
    )
    df_exploded['date'] = pd.to_datetime(df_exploded['date'], errors='coerce')

    # Prepara para o agrupamento semanal
    df_plot = df_exploded.dropna(subset=["macrotopic_pred"]).copy()
    df_plot["date"] = pd.to_datetime(df_plot["date"], errors="coerce")
    df_plot["week"] = df_plot["date"].dt.to_period("W").astype(str)

    # Agrupa por semana e macrotópico, calculando a toxicidade média
    grouped_data = (
        df_plot.groupby(["week", "macrotopic_pred"])["perspective_toxicity"]
        .mean()
        .reset_index()
    )
    return grouped_data

# --- 2. GERAÇÃO DO GRÁFICO PLOTLY ---
def create_interactive_plot(df):
    """Cria o gráfico de linhas interativo usando Plotly Express com zoom inicial e ajustes de legenda."""
    
    if df.empty:
        print("Nenhum dado para plotar.")
        return None
        
    fig = px.line(
        df,
        x="week",
        y="perspective_toxicity",
        color="macrotopic_pred",
        title="Evolução Semanal da Toxicidade por Macrotópico (Interativo)",
        color_discrete_map=MACRO_COLORS, 
        labels={
            "week": "Semana", 
            "perspective_toxicity": "Toxicidade Média", 
            "macrotopic_pred": "Macrotópico"
        },
        height=600
    )

    # 1. Aplicar Zoom no Eixo Y (Toxicidade)
    # Define o alcance inicial do eixo Y de 0.0 até 0.2 (ou o valor que você preferir)
    fig.update_yaxes(
        range=[0.0, 0.2],  # Limite inferior 0.0, limite superior 0.2
        title_text="Toxicidade Média"
    )

    # 2. Ajustes de Layout e Interatividade
    fig.update_traces(mode='lines+markers')
    
    # 3. Rotação e Posicionamento da Legenda
    fig.update_layout(
        hovermode="x unified",
        legend_title="Clique na legenda para ligar/desligar tópicos",
        legend=dict(
            # Posiciona a legenda fora do gráfico, no canto superior direito
            yanchor="top",
            y=1.0,
            xanchor="left",
            x=1.02,
            # Rotação do texto da legenda
            itemwidth=30, # Ajusta a largura para que o texto não fique muito longo
            # O Plotly não tem uma propriedade 'rotation' direta para itens da legenda,
            # mas o posicionamento externo (acima) já melhora a leitura horizontal.
            # Se for necessário ajustar a fonte:
            font=dict(
                size=10, 
            )
        )
    )

    return fig

# --- 3. EXECUÇÃO PRINCIPAL ---
if __name__ == "__main__":
    
    # Certifique-se de que a pasta 'data' e o arquivo 'macrotopics_pred.csv' existam.
    grouped_df = preprocess_data(DATA_PATH)
    
    if grouped_df is not None:
        plot_figure = create_interactive_plot(grouped_df)
        
        if plot_figure:
            # Salva o gráfico em um arquivo HTML autônomo.
            # O argumento 'full_html=True' é desnecessário aqui, pois
            # 'write_html' salva por padrão como um arquivo autônomo.
            plot_figure.write_html(
                file=OUTPUT_FILE,
                include_plotlyjs='cdn',  # Garante que o Plotly.js seja carregado via CDN (funciona melhor)
                auto_open=False
            )
            print(f"\n✅ Sucesso! Gráfico interativo salvo como: {OUTPUT_FILE}")
            print("Você pode enviar este arquivo HTML para qualquer pessoa. Ela só precisa de um navegador para abri-lo.")


✅ Sucesso! Gráfico interativo salvo como: analise_temporal_interativa.html
Você pode enviar este arquivo HTML para qualquer pessoa. Ela só precisa de um navegador para abri-lo.


In [2]:
import pandas as pd

# --- PASSO 0: DIAGNÓSTICO (Verifique se todos aparecem aqui) ---
print("Tópicos encontrados no DF atual:", df['macrotopic_pred'].unique())
# Se aqui só aparecerem 7 tópicos, você precisa recarregar seu dataframe completo 
# ou usar a variável correta (ex: df_full) antes de seguir.

# --- PASSO 1: CÁLCULO (Usando o DF completo) ---
df_mean = df.groupby('macrotopic_pred')[tox_cols].mean()
df_std  = df.groupby('macrotopic_pred')[tox_cols].std()

# Ordenar por toxicidade decrescente
df_mean = df_mean.sort_values(by='perspective_toxicity', ascending=False)
df_std = df_std.reindex(df_mean.index)

# Criar texto "Média ± Desvio"
df_tabela = df_mean.round(3).astype(str) + " ± " + df_std.round(3).astype(str)

# Limpeza e Ajuste dos Nomes (Remove underscores e Capitaliza)
# IMPORTANTE: Isso muda 'science_and_technology' para 'Science and technology' se for o caso
df_tabela.index = df_tabela.index.str.replace('_', ' ').str.capitalize()
df_tabela.columns = [c.replace('perspective_', '').replace('_', ' ').capitalize() for c in df_tabela.columns]

# --- PASSO 2: DEFINIÇÃO DE CORES (Atualizada para todos os tópicos) ---

politicos = [
    'American politics', 
    'War', 
    'Foreign policy', 
    'Crime, security and justice'
]

nao_politicos = [
    'Health', 
    'Hobbies', 
    'Religion', 
    'Entertainment', 
    'Technology',            # Verifique se no seu print saiu 'Technology' ou 'Science and technology'
    'Science and technology', # Adicionei as duas opções por garantia
    'Environment'
]

def colorir_fundo_index(valor):
    """Pinta o FUNDO do nome do tópico"""
    if valor in politicos:
        return 'background-color: #2ca02c; color: white; font-weight: bold;' # Verde
    elif valor in nao_politicos:
        return 'background-color: #1f77b4; color: white; font-weight: bold;' # Azul
    return '' # Se sobrar algum tópico estranho, fica sem cor

# --- PASSO 3: ESTILIZAÇÃO ---
styled_table = (df_tabela.style
    .set_caption("Média e Desvio Padrão da Toxicidade (Todos os Tópicos)")
    
    # Aplica cores no Índice (Nomes)
    .map_index(colorir_fundo_index, axis=0)
    
    # Aplica fundo branco nos Dados
    .set_properties(**{
        'background-color': 'white',  
        'color': 'black',             
        'border-color': '#d3d3d3',    
        'text-align': 'center',
        'padding': '8px'
    })
    
    # Estilo do Cabeçalho
    .set_table_styles([
        {'selector': 'th.col_heading', 'props': [
            ('background-color', '#1f77b4'), 
            ('color', 'white'), 
            ('font-weight', 'bold'),
            ('text-align', 'center')
        ]},
        {'selector': 'caption', 'props': [
            ('font-size', '14px'), 
            ('font-weight', 'bold'),
            ('color', 'black')
        ]}
    ])
)

display(styled_table)

NameError: name 'df' is not defined

In [ ]:
import pandas as pd
import ast
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.lines import Line2D
from ipywidgets import interactive, SelectMultiple
from IPython.display import display

# A função de plotagem que você forneceu:
def plot_weekly_video_count_macros(df, selected_macros, title_label="All Data"):
    """
    Plota o número semanal de vídeos para os macrotópicos selecionados.
    Inclui filtro de tópicos e ajuste de labels do eixo X.
    """
    # 1. Filtra os dados apenas para os macrotópicos selecionados
    if not selected_macros:
        print("Selecione pelo menos um macrotópico para plotar.")
        return
    df = df[df["macrotopic_pred"].isin(selected_macros)].copy()

    # 2. Pré-processamento e criação de semana numérica
    df = df.dropna(subset=["macrotopic_pred"]).copy()
    df["date"] = pd.to_datetime(df["date"], errors="coerce")
    
    # 🚨 Garante que não haja NaT (data inválida) antes de calcular a semana
    df = df.dropna(subset=["date"])

    df["week_start"] = df["date"].dt.to_period("W").dt.start_time
    df = df.dropna(subset=["week_start"])

    # Lista de semanas e numeração
    week_list = sorted(df["week_start"].unique())
    week_map = {w: i+1 for i, w in enumerate(week_list)}
    df["week_num"] = df["week_start"].map(week_map)

    macro_colors = {
        "American Politics": "#6AF5FF",
        "Foreign Policy": "#76FF7F",
        "Crime, Security and Justice": "#FF66CC",
        "Environment": "#522222",
        "Health": "#14712A",
        "Entertainment": "#ECF988",
        "Hobbies": "#DA0000",
        "Religion": "#962EAA",
        "Technology": "#0000FF",
        "War": "#5C5C61",
        "Others": "#FF9900"
    }

    # 3. Agrupamento e Contagem
    grouped = (
        df.groupby(["week_num", "macrotopic_pred"])["video_id"]
        .count() 
        .reset_index(name="video_count")
    )

    macro_categories = sorted(grouped["macrotopic_pred"].unique())

    # 4. Plotagem (Matplotlib/Seaborn)
    sns.set(style="whitegrid")
    plt.figure(figsize=(18, 9))

    for macro in macro_categories:
        m_df = grouped[grouped["macrotopic_pred"] == macro]
        plt.plot(
            m_df["week_num"],
            m_df["video_count"],
            marker="o",
            label=macro,
            color=macro_colors.get(macro, "#333333")
        )

    # 5. Configuração dos labels do eixo X (complexo, mantido como fornecido)
    n = len(week_list)
    mid = n // 2
    
    # Define a data da eleição
    election_date = pd.Timestamp("2024-11-05")
    # Encontra a semana mais próxima à data da eleição
    if week_list:
        election_week = min(week_list, key=lambda x: abs(x - election_date))
    else:
        election_week = None # Caso a lista esteja vazia

    labels = []
    for i, w in enumerate(week_list, start=1):
        is_key_week = (i == 1 or i == mid or i == n or w == election_week)
        if is_key_week and w:
            labels.append(w.strftime("%d/%b/%Y"))
        else:
            labels.append(f"Week {i}")
            
    # Assegura que o número de ticks e labels seja o mesmo
    if len(week_list) > 0:
        plt.xticks(
            ticks=list(range(1, n+1)),
            labels=labels,
            rotation=90 
        )

    plt.title(f"Weekly Number of Videos — {title_label}")
    plt.xlabel("Weeks")
    plt.ylabel("Number of Videos")
    plt.ylim(0, 20000)

    # 6. Legenda
    handles = [
        Line2D([0], [0], marker="s", linestyle="", 
                color=macro_colors[m], markersize=12, label=m)
        for m in selected_macros if m in macro_colors # Usa a lista filtrada
    ]

    plt.legend(handles=handles, title="Macrotopic",
                bbox_to_anchor=(1.05, 1), loc="upper left")

    plt.tight_layout()
    plt.show()

In [ ]:
# --- CÓDIGO DE PREPARAÇÃO DE DADOS (Você deve rodar este bloco) ---

# Importa ast para avaliar strings como listas/dicionários
import ast

# Assume que seu arquivo de dados está aqui
DATA_PATH = "data/macrotopics_pred.csv" 
df_final = pd.read_csv(DATA_PATH) 

def parse_occurrences(x):
    try:
        # Usa ast.literal_eval para converter a string para uma estrutura Python (lista de dicts)
        return ast.literal_eval(x)
    except:
        return []

df_final['occ_list'] = df_final['occurrences'].apply(parse_occurrences)

# 1. Explode o DataFrame (cria uma linha por ocorrência de data)
df_exploded = df_final.explode('occ_list', ignore_index=True)

# 2. Extrai a data
df_exploded['date'] = df_exploded['occ_list'].apply(
    lambda x: x.get('date') if isinstance(x, dict) and 'date' in x else None
)
df_exploded['date'] = pd.to_datetime(df_exploded['date'], errors='coerce')

# 3. Identifica todos os macrotópicos únicos disponíveis
all_macro_categories = sorted(df_exploded["macrotopic_pred"].dropna().unique())

# O DataFrame 'df_exploded' é o input necessário para a função 'plot_weekly_video_count_macros'

In [ ]:
# 1. Cria o widget de seleção múltipla
topic_selector = SelectMultiple(
    options=all_macro_categories,
    value=all_macro_categories,  # Começa com todos selecionados
    description='Tópicos:',
    disabled=False,
    rows=min(len(all_macro_categories), 15), # Limita o tamanho da caixa
)

# 2. Conecta o widget à função de plotagem usando o DataFrame preparado
# O argumento 'selected_macros' na função de plotagem será mapeado para o valor do 'topic_selector'
interactive_plot = interactive(
    plot_weekly_video_count_macros,
    df=fixed(df_exploded), # Fixa o DataFrame, ele não muda com a interação
    selected_macros=topic_selector,
    title_label="Video Count Analysis" 
)

# 3. Exibe o widget e o gráfico
display(interactive_plot)

NameError: name 'fixed' is not defined